# Named Entity Recognition (NER) from News Articles
Dataset: CoNLL-2003 (Kaggle - `alaakhaled/conll003-englishversion`)

Identifying named entities (people, locations, organizations) in news text, using both a **rule-based** approach and a **pretrained model-based** approach (spaCy), then comparing them against the dataset's gold-standard labels.

## Download the dataset from Kaggle

In [1]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("alaakhaled/conll003-englishversion")

print("Path to dataset files:", path)

100%|██████████| 960k/960k [00:00<00:00, 107MB/s]

Extracting files...
Path to dataset files: /root/.cache/kagglehub/datasets/alaakhaled/conll003-englishversion/versions/1


In [2]:
!ls /root/.cache/kagglehub/datasets/alaakhaled/conll003-englishversion/versions/1

metadata  test.txt  train.txt  valid.txt


## Install & Import libraries
Installing spaCy and two of its pretrained English pipelines (`en_core_web_sm` and `en_core_web_md`, needed later for the bonus model comparison), plus the other libraries used throughout the notebook.

In [3]:
!pip install -q spacy
!python -m spacy download en_core_web_sm -q
!python -m spacy download en_core_web_md -q

import glob
import os
import re
import random
from collections import Counter

import pandas as pd
import matplotlib.pyplot as plt

import spacy
from spacy import displacy

nlp_sm = spacy.load("en_core_web_sm")
nlp_md = spacy.load("en_core_web_md")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 86.5 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 33.5/33.5 MB 63.9 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_md')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


## Parse the CoNLL-2003 dataset
The dataset ships as `train.txt` / `valid.txt` / `test.txt` in the standard CoNLL-2003 format: one token per line (`word  POS-tag  chunk-tag  NER-tag`), sentences separated by blank lines, and `-DOCSTART-` lines marking document boundaries (not real tokens). NER tags use the IOB2 scheme, e.g. `B-PER`, `I-PER`, `B-ORG`, `B-LOC`, `B-MISC`, `O`.

We parse the file into a list of sentences, each as a list of `(token, ner_tag)` pairs.

In [4]:
def parse_conll_file(filepath):
    sentences = []
    current = []
    with open(filepath, encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if not line or line.startswith('-DOCSTART-'):
                if current:
                    sentences.append(current)
                    current = []
                continue
            parts = line.split()
            token, ner_tag = parts[0], parts[-1]
            current.append((token, ner_tag))
    if current:
        sentences.append(current)
    return sentences

data_dir = [f for f in glob.glob(os.path.join(path, "**"), recursive=True) if f.endswith('test.txt')][0]
data_dir = os.path.dirname(data_dir)

train_sentences = parse_conll_file(os.path.join(data_dir, 'train.txt'))
test_sentences = parse_conll_file(os.path.join(data_dir, 'test.txt'))

print("Train sentences:", len(train_sentences))
print("Test sentences :", len(test_sentences))
print("\nExample sentence (token, tag) pairs:")
print(test_sentences[1][:10])

Train sentences: 14041
Test sentences : 3453

Example sentence (token, tag) pairs:
[('Nadim', 'B-PER'), ('Ladki', 'I-PER')]


## Reconstruct raw text and gold entity spans
spaCy and any rule-based approach need plain text, not pre-tokenized IOB tags. For each sentence we:
- Join the tokens back into a single string (tracking each token's character offset so we can align gold labels later).
- Convert the IOB2 tags into entity spans: `(start_char, end_char, entity_type)`.

CoNLL uses 4 entity types: `PER`, `ORG`, `LOC`, `MISC`. Since spaCy's pretrained models have no direct equivalent for `MISC` (it's a catch-all category for things like nationalities, events, and other named entities that don't fit the other three), we drop `MISC` entities and evaluate only on `PER`, `ORG`, and `LOC` — the three types both approaches can meaningfully be compared on.

In [5]:
def sentence_to_text_and_spans(sentence):
    text = ""
    spans = []
    current_type = None
    current_start = None

    for token, tag in sentence:
        start = len(text)
        text += token + " "
        end = start + len(token)

        if tag.startswith('B-'):
            if current_type is not None:
                spans.append((current_start, prev_end, current_type))
            current_type = tag[2:]
            current_start = start
        elif tag.startswith('I-') and current_type == tag[2:]:
            pass  # continue the current entity
        else:  # 'O' or a mismatched I- tag
            if current_type is not None:
                spans.append((current_start, prev_end, current_type))
            current_type = None
            current_start = None
        prev_end = end

    if current_type is not None:
        spans.append((current_start, prev_end, current_type))

    text = text.strip()
    # keep only the 3 comparable types
    spans = [s for s in spans if s[2] in ('PER', 'ORG', 'LOC')]
    return text, spans

test_examples = [sentence_to_text_and_spans(s) for s in test_sentences]
test_examples = [(text, spans) for text, spans in test_examples if text and spans]

print("Usable test sentences (with at least one PER/ORG/LOC entity):", len(test_examples))
print("\nExample:")
text, spans = test_examples[0]
print("TEXT :", text)
print("SPANS:", [(text[s:e], t) for s, e, t in spans])

Usable test sentences (with at least one PER/ORG/LOC entity): 2541

Example:
TEXT : SOCCER - JAPAN GET LUCKY WIN , CHINA IN SURPRISE DEFEAT .
SPANS: [('JAPAN', 'LOC'), ('CHINA', 'PER')]


## Model-based NER: pretrained spaCy pipeline
Running spaCy's pretrained `en_core_web_sm` model on the same raw sentences. spaCy's entity types don't match CoNLL's naming exactly, so we map them to the comparable set:
- `PERSON` -> `PER`
- `ORG` -> `ORG`
- `GPE` and `LOC` (spaCy separates countries/cities from other locations; CoNLL doesn't) -> `LOC`

Any other spaCy entity type (dates, money, products, etc.) is ignored for this comparison.

In [6]:
SPACY_TO_CONLL = {'PERSON': 'PER', 'ORG': 'ORG', 'GPE': 'LOC', 'LOC': 'LOC'}

def model_based_ner(text, nlp):
    doc = nlp(text)
    spans = []
    for ent in doc.ents:
        if ent.label_ in SPACY_TO_CONLL:
            spans.append((ent.start_char, ent.end_char, SPACY_TO_CONLL[ent.label_]))
    return spans

# Quick look at a few predictions
for text, gold_spans in test_examples[:3]:
    pred_spans = model_based_ner(text, nlp_sm)
    print("TEXT:", text)
    print("GOLD:", [(text[s:e], t) for s, e, t in gold_spans])
    print("PRED:", [(text[s:e], t) for s, e, t in pred_spans])
    print()

TEXT: SOCCER - JAPAN GET LUCKY WIN , CHINA IN SURPRISE DEFEAT .
GOLD: [('JAPAN', 'LOC'), ('CHINA', 'PER')]
PRED: [('DEFEAT', 'ORG')]

TEXT: Nadim Ladki
GOLD: [('Nadim Ladki', 'PER')]
PRED: [('Nadim Ladki', 'PER')]

TEXT: AL-AIN , United Arab Emirates 1996-12-06
GOLD: [('AL-AIN', 'LOC'), ('United Arab Emirates', 'LOC')]
PRED: [('AL-AIN', 'ORG'), ('United Arab Emirates', 'LOC')]



## Bonus: Visualize entities with displaCy
Rendering the pretrained model's predictions directly on top of the text with spaCy's built-in visualizer.

In [7]:
sample_text = test_examples[5][0]
doc = nlp_sm(sample_text)
displacy.render(doc, style='ent', jupyter=True)

## Rule-based NER approach
A simple, hand-written heuristic with no machine learning at all: any run of consecutive capitalized words (that aren't the first word of the sentence, to avoid flagging ordinary sentence-starting capitals) is treated as a candidate entity. This is a classic, simple rule-based technique, but it can only detect entity **boundaries** — it has no way to know whether a capitalized phrase is a person, an organization, or a location, so it assigns a single generic type to everything it finds.

In [8]:
def rule_based_ner(text):
    tokens = text.split()
    spans = []
    pos = 0
    i = 0
    offsets = []
    cursor = 0
    for tok in tokens:
        start = text.index(tok, cursor)
        end = start + len(tok)
        offsets.append((start, end))
        cursor = end

    i = 0
    while i < len(tokens):
        if i > 0 and tokens[i][0:1].isupper() and tokens[i].isalpha():
            j = i
            while j < len(tokens) and tokens[j][0:1].isupper() and tokens[j].isalpha():
                j += 1
            span_start = offsets[i][0]
            span_end = offsets[j - 1][1]
            spans.append((span_start, span_end, 'ENTITY'))
            i = j
        else:
            i += 1
    return spans

# Quick look at the same examples
for text, gold_spans in test_examples[:3]:
    pred_spans = rule_based_ner(text)
    print("TEXT:", text)
    print("GOLD:", [(text[s:e], t) for s, e, t in gold_spans])
    print("RULE:", [(text[s:e]) for s, e, t in pred_spans])
    print()

TEXT: SOCCER - JAPAN GET LUCKY WIN , CHINA IN SURPRISE DEFEAT .
GOLD: [('JAPAN', 'LOC'), ('CHINA', 'PER')]
RULE: ['JAPAN GET LUCKY WIN', 'CHINA IN SURPRISE DEFEAT']

TEXT: Nadim Ladki
GOLD: [('Nadim Ladki', 'PER')]
RULE: ['Ladki']

TEXT: AL-AIN , United Arab Emirates 1996-12-06
GOLD: [('AL-AIN', 'LOC'), ('United Arab Emirates', 'LOC')]
RULE: ['United Arab Emirates']



## Evaluate: rule-based vs. model-based
Since the rule-based approach can't predict entity *types*, we evaluate both approaches on **boundary detection only**: does the predicted span's character range exactly match a gold entity's range, regardless of type? This is a fair comparison of what rule-based matching is actually capable of, run over the full test set.

In [9]:
def evaluate_boundaries(examples, predict_fn, n=None):
    tp, fp, fn = 0, 0, 0
    for text, gold_spans in (examples if n is None else examples[:n]):
        gold_set = {(s, e) for s, e, t in gold_spans}
        pred_spans = predict_fn(text)
        pred_set = {(s, e) for s, e, t in pred_spans}

        tp += len(gold_set & pred_set)
        fp += len(pred_set - gold_set)
        fn += len(gold_set - pred_set)

    precision = tp / (tp + fp) if (tp + fp) else 0
    recall = tp / (tp + fn) if (tp + fn) else 0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) else 0
    return precision, recall, f1

rule_p, rule_r, rule_f1 = evaluate_boundaries(test_examples, rule_based_ner)
model_p, model_r, model_f1 = evaluate_boundaries(test_examples, lambda t: model_based_ner(t, nlp_sm))

print(f"Rule-based  -> Precision: {rule_p:.3f} | Recall: {rule_r:.3f} | F1: {rule_f1:.3f}")
print(f"Model-based -> Precision: {model_p:.3f} | Recall: {model_r:.3f} | F1: {model_f1:.3f}")

Rule-based  -> Precision: 0.636 | Recall: 0.617 | F1: 0.627
Model-based -> Precision: 0.846 | Recall: 0.677 | F1: 0.752


## Compare two different spaCy models
Comparing `en_core_web_sm` (small, fast) against `en_core_web_md` (larger, uses word vectors) on the same boundary-detection evaluation over the full test set, and on entity type accuracy for the entities each one gets right.

In [10]:
def evaluate_with_types(examples, nlp, n=None):
    boundary_tp, fp, fn = 0, 0, 0
    type_correct = 0
    for text, gold_spans in (examples if n is None else examples[:n]):
        gold_dict = {(s, e): t for s, e, t in gold_spans}
        pred_spans = model_based_ner(text, nlp)
        pred_dict = {(s, e): t for s, e, t in pred_spans}

        for span in gold_dict:
            if span in pred_dict:
                boundary_tp += 1
                if pred_dict[span] == gold_dict[span]:
                    type_correct += 1
            else:
                fn += 1
        for span in pred_dict:
            if span not in gold_dict:
                fp += 1

    precision = boundary_tp / (boundary_tp + fp) if (boundary_tp + fp) else 0
    recall = boundary_tp / (boundary_tp + fn) if (boundary_tp + fn) else 0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) else 0
    type_acc = type_correct / boundary_tp if boundary_tp else 0
    return precision, recall, f1, type_acc

sm_p, sm_r, sm_f1, sm_type_acc = evaluate_with_types(test_examples, nlp_sm)
md_p, md_r, md_f1, md_type_acc = evaluate_with_types(test_examples, nlp_md)

print("=== en_core_web_sm ===")
print(f"Precision: {sm_p:.3f} | Recall: {sm_r:.3f} | F1: {sm_f1:.3f} | Type accuracy (on correct boundaries): {sm_type_acc:.3f}")

print("\n=== en_core_web_md ===")
print(f"Precision: {md_p:.3f} | Recall: {md_r:.3f} | F1: {md_f1:.3f} | Type accuracy (on correct boundaries): {md_type_acc:.3f}")

=== en_core_web_sm ===
Precision: 0.846 | Recall: 0.677 | F1: 0.752 | Type accuracy (on correct boundaries): 0.815

=== en_core_web_md ===
Precision: 0.834 | Recall: 0.738 | F1: 0.783 | Type accuracy (on correct boundaries): 0.794


## Final comparison

In [11]:
print("=== Final Comparison (boundary-detection F1, full test set) ===")
print(f"Rule-based approach     : {rule_f1:.3f}")
print(f"en_core_web_sm (model)  : {sm_f1:.3f}")
print(f"en_core_web_md (model)  : {md_f1:.3f}")

=== Final Comparison (boundary-detection F1, full test set) ===
Rule-based approach     : 0.627
en_core_web_sm (model)  : 0.752
en_core_web_md (model)  : 0.783
